In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/GPT-from-Scratch/notebooks/"
os.chdir(PROJECT_DIR)
print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/GPT-from-Scratch/notebooks


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer

In [3]:
tokenizer = Tokenizer.from_file("../src/tokenizer/arabic_bpe_tokenizer.json")

with open("../data/pretrain/data.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

token_ids = tokenizer.encode(raw_text).ids

print("Total token count:", len(token_ids))
print("First 50 token ids:", token_ids[:50])

Total token count: 320053
First 50 token ids: [4382, 814, 366, 851, 6, 616, 4072, 7, 4382, 814, 366, 851, 6, 616, 4072, 7, 1200, 223, 277, 296, 1860, 2483, 866, 723, 3447, 439, 206, 2315, 105, 688, 252, 278, 1565, 549, 1070, 585, 156, 524, 520, 288, 4905, 711, 198, 1504, 169, 616, 1430, 156, 123, 224]


In [4]:
class GPTDataset(Dataset):
    def __init__(self, token_ids, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk, dtype=torch.long))
            self.target_ids.append(torch.tensor(target_chunk, dtype=torch.long))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [5]:
context_length = 128
stride = 64
batch_size = 8

dataset = GPTDataset(token_ids, max_length=context_length, stride=stride)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

print("Number of training samples:", len(dataset))
print("Number of batches:", len(dataloader))

Number of training samples: 4999
Number of batches: 624


In [6]:
for x, y in dataloader:
    print("Input batch shape:", x.shape)
    print("Target batch shape:", y.shape)
    print("\nFirst input sample:\n", x[0][:20])
    print("\nFirst target sample:\n", y[0][:20])
    break

Input batch shape: torch.Size([8, 128])
Target batch shape: torch.Size([8, 128])

First input sample:
 tensor([  83,  802, 1080,  189,  620, 1643,  556,  246,  501,  195,  997,  539,
        3234, 3839,  614,  188,  429, 2049, 1713,  170])

First target sample:
 tensor([ 802, 1080,  189,  620, 1643,  556,  246,  501,  195,  997,  539, 3234,
        3839,  614,  188,  429, 2049, 1713,  170,  482])


In [7]:
sample_input, sample_target = dataset[0]

print("Input first 20 ids:")
print(sample_input[:20].tolist())

print("\nTarget first 20 ids:")
print(sample_target[:20].tolist())

Input first 20 ids:
[4382, 814, 366, 851, 6, 616, 4072, 7, 4382, 814, 366, 851, 6, 616, 4072, 7, 1200, 223, 277, 296]

Target first 20 ids:
[814, 366, 851, 6, 616, 4072, 7, 4382, 814, 366, 851, 6, 616, 4072, 7, 1200, 223, 277, 296, 1860]
